# Course 1, Week 2 — The PyTorch Workflow

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #2](https://github.com/majorgilles/pytorch_for_deep_learning/issues/2)

**Focus:** Build the training loop: model, loss, optimizer, gradients, evaluation, and saving.


## From tensor fundamentals to image classification

The previous week introduced tensors and the basic PyTorch training loop. This week applies that foundation to a more demanding task: recognizing handwritten characters from images. A character classifier predicts one class for each segmented image; those predictions can then be assembled into readable text.

### Why images require a stronger workflow

An image contains many more input values than the small tabular examples used previously. A grayscale MNIST image has height $28$ and width $28$, giving $28 \times 28 = 784$ pixel values. A batch is commonly represented with shape $[B, C, H, W]$, where:

- $B$ is the number of images in the batch.
- $C=1$ is the grayscale channel.
- $H=W=28$ are the image dimensions.

The classifier must map each image to one of ten digit classes, $0$ through $9$. MNIST is a useful starting point because its images are consistently sized, centered, labeled, and small enough to train quickly.

### Learning path

The image-classification workflow introduces several tools that become essential as datasets and models grow:

1. Load and batch image data efficiently.
2. Build multi-layer models with more flexibility than a simple linear predictor.
3. Choose losses and optimizers suited to classification.
4. Inspect the training process through predictions, losses, and gradients.
5. Move tensors and models between CPU and GPU devices safely.

MNIST provides a compact environment for practicing this complete pipeline before progressing to convolutional neural networks and more varied handwriting.


## Loading data the PyTorch way

The machine learning pipeline still begins with data, but image datasets make memory management more important. Loading every sample into RAM may work for a small delivery table, but it stops scaling as the number and size of the records grow. The practical rule is simple: **load only the data needed for the current batch**.

PyTorch organizes this work with three tools that fit together:

1. **Transforms** prepare each sample as it is loaded.
2. **Dataset** describes how to find, load, and count individual samples.
3. **DataLoader** requests samples from the dataset and serves them in batches.

The resulting flow is:

$$\text{stored sample} \longrightarrow \text{transform} \longrightarrow \text{dataset} \longrightarrow \text{batch} \longrightarrow \text{model}.$$

### 1. Transform each sample

`transforms.Compose` applies several operations in sequence. For common image inputs, `ToTensor` converts the image to a tensor and scales unsigned 8-bit pixel values from $[0,255]$ to $[0,1]$. `Normalize` then transforms each channel using

$$x' = \frac{x-\mu}{\sigma},$$

which can give optimization a better-scaled input. More advanced augmentation and preprocessing techniques come later.

### 2. Address samples through a dataset

A `Dataset` provides two essential behaviors:

- `len(dataset)` reports the number of samples.
- `dataset[index]` loads one sample and its label.

The sample can be read from storage and transformed only when requested rather than being preloaded with the entire dataset. Built-in datasets such as MNIST also support selecting the training or test split and downloading missing files. Custom dataset classes use the same interface.

### 3. Serve batches with a data loader

A `DataLoader` groups requested samples into manageable batches. `batch_size` controls how many examples are returned at once, while `shuffle=True` changes the training order between passes through the dataset. For MNIST, a batch of 64 images has shape $[64,1,28,28]$ and its labels have shape $[64]$.


In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Apply each preprocessing step when an image is requested.
image_transform: transforms.Compose = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,), std=(0.5,)),
    ]
)

# The dataset identifies samples; it does not place every image in one tensor.
train_dataset: datasets.MNIST = datasets.MNIST(
    root="data", train=True, download=True, transform=image_transform
)

# The loader retrieves one shuffled batch at a time.
train_loader: DataLoader = DataLoader(
    train_dataset, batch_size=64, shuffle=True
)

batch: tuple[torch.Tensor, torch.Tensor] = next(iter(train_loader))
images, labels = batch
print(f"image batch: {images.shape}")
print(f"label batch: {labels.shape}")

image batch: torch.Size([64, 1, 28, 28])
label batch: torch.Size([64])


This pattern scales from delivery records to image collections: transform one sample, let the dataset retrieve it, and let the data loader assemble only the next batch. With the data pipeline in place, the next step is to examine how losses, gradients, and optimizers train the model.


## Building models with `nn.Module`

`nn.Sequential` is convenient when data passes through layers in one fixed order. A custom `nn.Module` expresses the same computation with more control and is the standard pattern for models with branches, skip connections, or other custom behavior.

Every custom module has two central parts:

1. **`__init__` defines the layers.** Calling `super().__init__()` first lets PyTorch register their learnable weights and biases.
2. **`forward` defines the data flow.** It describes the order in which those layers process an input tensor.

Call the module as `model(inputs)`, not `model.forward(inputs)`. The module call invokes `forward` while preserving PyTorch's hooks and other internal bookkeeping.


In [2]:
from torch import nn, optim


class DigitClassifier(nn.Module):
    """Map image batches of shape $[B, 1, 28, 28]$ to logits of shape $[B, 10]$."""

    def __init__(self) -> None:
        super().__init__()
        self.flatten: nn.Flatten = nn.Flatten()
        self.hidden: nn.Linear = nn.Linear(28 * 28, 128)
        self.activation: nn.ReLU = nn.ReLU()
        self.output: nn.Linear = nn.Linear(128, 10)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Return class logits for inputs with shape $[B, 1, 28, 28]$."""
        flattened: torch.Tensor = self.flatten(inputs)
        hidden: torch.Tensor = self.activation(self.hidden(flattened))
        return self.output(hidden)


model: DigitClassifier = DigitClassifier()
logits: torch.Tensor = model(images)
print(f"input: {images.shape} -> logits: {logits.shape}")

input: torch.Size([64, 1, 28, 28]) -> logits: torch.Size([64, 10])


## Training in the correct order

A standard PyTorch training step follows the same sequence for every batch:

1. `optimizer.zero_grad()` clears gradients accumulated by earlier batches.
2. `model(inputs)` performs the forward pass.
3. The loss function compares the logits with the correct labels.
4. `loss.backward()` computes parameter gradients for the current batch.
5. `optimizer.step()` updates the parameters using those gradients.

The order matters even when incorrect code does not immediately raise an exception. Stepping before backpropagation uses stale gradients, clearing gradients after backpropagation discards the new gradients, and clearing only once outside the loop causes gradients from multiple batches to accumulate.


In [3]:
model.train()
loss_function: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
optimizer: optim.SGD = optim.SGD(model.parameters(), lr=0.01)

optimizer.zero_grad()
logits = model(images)
loss: torch.Tensor = loss_function(logits, labels)
loss.backward()
optimizer.step()

print(f"training loss: {loss.item():.4f}")

training loss: 2.2926


## Evaluating on unseen data

Evaluation checks whether the model generalizes beyond its training examples. Two PyTorch tools prepare the model for this phase:

- `model.eval()` switches layers with training-specific behavior, such as dropout and batch normalization, into evaluation mode. It does **not** calculate a score.
- `torch.no_grad()` disables gradient tracking, reducing unnecessary memory and computation.

For classification, accuracy is the fraction of correct predictions:

$$\operatorname{accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}.$$

Predictions must be evaluated on a separate test or validation split. Measuring only the training data cannot reveal whether the model learned a reusable pattern or merely fit examples it already saw. Call `model.train()` again before resuming training.


In [4]:
test_dataset: datasets.MNIST = datasets.MNIST(
    root="data", train=False, download=True, transform=image_transform
)
test_loader: DataLoader = DataLoader(
    test_dataset, batch_size=64, shuffle=False
)
test_batch: tuple[torch.Tensor, torch.Tensor] = next(iter(test_loader))
test_images: torch.Tensor = test_batch[0]
test_labels: torch.Tensor = test_batch[1]

model.eval()
with torch.no_grad():
    test_logits: torch.Tensor = model(test_images)
    predictions: torch.Tensor = test_logits.argmax(dim=1)

correct: int = int((predictions == test_labels).sum().item())
total: int = test_labels.numel()
accuracy: float = correct / total
print(f"accuracy on one unseen batch: {accuracy:.1%}")

# Restore training behavior before any further optimization.
model.train()

accuracy on one unseen batch: 3.1%


DigitClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (hidden): Linear(in_features=784, out_features=128, bias=True)
  (activation): ReLU()
  (output): Linear(in_features=128, out_features=10, bias=True)
)

## Loss: measuring the model's error

Three lines appear throughout PyTorch training because they form a repeating **measure → diagnose → update** cycle:

1. `loss = loss_function(predictions, targets)` **measures** how wrong the predictions are.
2. `loss.backward()` **diagnoses** how each learnable parameter contributed to that error by calculating gradients.
3. `optimizer.step()` **updates** each parameter using those gradients.

The loss compresses all prediction errors into one scalar objective. Training aims to reduce that value, but the correct way to measure error depends on the task.

### Mean squared error for regression

Regression predicts continuous quantities such as delivery time, temperature, distance, or price. For predictions $\hat{y}_i$ and targets $y_i$, mean squared error is

$$\operatorname{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2.$$

Averaging signed errors is not enough: positive and negative mistakes can cancel. Squaring ensures every mistake contributes and gives larger errors a stronger penalty.


In [5]:
delivery_predictions: torch.Tensor = torch.tensor([6.0, 3.0])
delivery_targets: torch.Tensor = torch.tensor([4.0, 5.0])
signed_errors: torch.Tensor = delivery_predictions - delivery_targets
mean_signed_error: torch.Tensor = signed_errors.mean()

mse_loss_function: nn.MSELoss = nn.MSELoss()
mse: torch.Tensor = mse_loss_function(delivery_predictions, delivery_targets)

print(f"signed errors: {signed_errors.tolist()}")
print(f"mean signed error: {mean_signed_error.item():.1f}")
print(f"mean squared error: {mse.item():.1f}")

signed errors: [2.0, -2.0]
mean signed error: 0.0
mean squared error: 4.0


### Cross-entropy loss for classification

Classification predicts a category such as a digit, animal, or word. The MNIST model returns ten **logits** with shape $[B,10]$, one score per digit. `nn.CrossEntropyLoss` compares those logits with integer class labels of shape $[B]$.

Softmax can convert logits into probabilities for interpretation, but do not apply it before `nn.CrossEntropyLoss`; PyTorch combines the required log-softmax and negative log-likelihood operations internally for numerical stability. The probability of the correct class depends on **every** logit because they all appear in the softmax denominator:

$$p_y = \frac{e^{z_y}}{\sum_j e^{z_j}}, \qquad L = -\log p_y.$$

Even when the correct-class logit $z_y$ stays fixed, increasing a wrong-class logit enlarges the denominator, steals probability from the correct class, and raises the loss. The next example prints that probability change directly.


In [6]:
target_digit: torch.Tensor = torch.tensor([3])

# Keep the correct-class logit at 0 and change only the competing class-7 logit.
high_wrong_logit: torch.Tensor = torch.zeros((1, 10))
high_wrong_logit[0, 7] = 5.0
low_wrong_logit: torch.Tensor = torch.zeros((1, 10))
low_wrong_logit[0, 7] = 1.0

cross_entropy: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
high_wrong_loss: torch.Tensor = cross_entropy(high_wrong_logit, target_digit)
low_wrong_loss: torch.Tensor = cross_entropy(low_wrong_logit, target_digit)
high_wrong_probabilities: torch.Tensor = torch.softmax(high_wrong_logit, dim=1)
low_wrong_probabilities: torch.Tensor = torch.softmax(low_wrong_logit, dim=1)

print("correct logit | wrong logit | correct probability | cross-entropy")
print(
    f"{high_wrong_logit[0, 3]:13.1f} | {high_wrong_logit[0, 7]:11.1f} | "
    f"{high_wrong_probabilities[0, 3]:19.4f} | {high_wrong_loss.item():13.3f}"
)
print(
    f"{low_wrong_logit[0, 3]:13.1f} | {low_wrong_logit[0, 7]:11.1f} | "
    f"{low_wrong_probabilities[0, 3]:19.4f} | {low_wrong_loss.item():13.3f}"
)

correct logit | wrong logit | correct probability | cross-entropy
          0.0 |         5.0 |              0.0064 |         5.059
          0.0 |         1.0 |              0.0853 |         2.461


### Choosing the loss

- Use **mean squared error** when the target is a continuous number.
- Use **cross-entropy loss** when exactly one class is correct for each example.
- Do not compare the raw values of different loss functions; they measure different objectives on different scales.
- Track the same loss over training and check that it generally decreases.

Loss completes the **measure** stage. Backpropagation and the optimizer use that measurement to diagnose parameter contributions and update the model.


## Backpropagation: diagnosing the error

After loss measures the error, `loss.backward()` calculates how sensitive that loss is to every trainable parameter. These sensitivities are **gradients**. For a parameter $\theta$, PyTorch computes

$$\frac{\partial L}{\partial \theta}.$$

A gradient communicates three useful facts:

- Its **sign** gives the local direction in which increasing the parameter would change the loss.
- Its **magnitude** measures how strongly a small parameter change would affect the loss.
- Its location in `parameter.grad` associates that diagnosis with the corresponding weight or bias.

The digit classifier already contains $101{,}770$ trainable parameters:

- Input-to-hidden weights: $784 \times 128 = 100{,}352$
- Hidden biases: $128$
- Hidden-to-output weights: $128 \times 10 = 1{,}280$
- Output biases: $10$

Autograd applies the chain rule through the recorded computation graph to calculate all of these gradients. Crucially, `backward()` only fills the `.grad` fields—it does **not** change the parameters.


In [7]:
parameter_counts: dict[str, int] = {
    name: parameter.numel() for name, parameter in model.named_parameters()
}
total_parameters: int = sum(parameter_counts.values())

for parameter_name, count in parameter_counts.items():
    print(f"{parameter_name:15} {count:>7,}")
print(f"{'total':15} {total_parameters:>7,}")
assert total_parameters == 101_770

hidden.weight   100,352
hidden.bias         128
output.weight     1,280
output.bias          10
total           101,770


## Optimizers: updating the parameters

The optimizer reads the gradients and performs the update. Stochastic gradient descent moves each parameter opposite its gradient:

$$\theta \leftarrow \theta - \eta\frac{\partial L}{\partial \theta},$$

where $\eta$ is the learning rate. For example, a gradient of $0.5$ and learning rate of $0.01$ produce an update of $-0.005$.

Learning-rate size controls the step:

- **Too small:** training makes progress very slowly.
- **Suitable:** loss descends steadily toward a useful solution.
- **Too large:** updates can overshoot, oscillate, or make the loss diverge.

SGD applies this direct rule. Adam also tracks moving averages of gradients and squared gradients so it can adapt the effective update for each parameter. Adam is often a useful starting point, but its learning rate is tuned differently from SGD; optimizer learning rates are not interchangeable defaults.


In [8]:
gradient_model: DigitClassifier = DigitClassifier()
gradient_optimizer: optim.SGD = optim.SGD(gradient_model.parameters(), lr=0.01)
gradient_loss_function: nn.CrossEntropyLoss = nn.CrossEntropyLoss()

gradient_optimizer.zero_grad()
weight_before: torch.Tensor = gradient_model.hidden.weight.detach().clone()
gradient_logits: torch.Tensor = gradient_model(images)
gradient_loss: torch.Tensor = gradient_loss_function(gradient_logits, labels)
gradient_loss.backward()

hidden_gradient: torch.Tensor | None = gradient_model.hidden.weight.grad
assert hidden_gradient is not None
weight_after_backward: torch.Tensor = gradient_model.hidden.weight.detach().clone()
print(f"gradient shape: {hidden_gradient.shape}")
print(f"gradient norm: {hidden_gradient.norm().item():.4f}")
print(f"weights changed by backward: {not torch.equal(weight_before, weight_after_backward)}")

gradient_optimizer.step()
weight_after_step: torch.Tensor = gradient_model.hidden.weight.detach()
update_size: float = (weight_after_step - weight_before).norm().item()
print(f"weights changed by step: {not torch.equal(weight_before, weight_after_step)}")
print(f"update norm: {update_size:.4f}")

gradient shape: torch.Size([128, 784])
gradient norm: 2.3240
weights changed by backward: False
weights changed by step: True
update norm: 0.0232


## Why gradients must be cleared

PyTorch **accumulates** gradients: each call to `backward()` adds its result to the existing `.grad` tensors. This behavior supports deliberate techniques such as combining gradients from several small batches, but ordinary training usually needs an independent diagnosis for every batch.

The complete sequence is therefore:

1. `optimizer.zero_grad()` clears the previous gradients.
2. `predictions = model(inputs)` performs the forward pass.
3. `loss = loss_function(predictions, targets)` measures error.
4. `loss.backward()` calculates and stores gradients.
5. `optimizer.step()` updates the parameters.

Loss **measures**, backpropagation **diagnoses**, and the optimizer **updates**. The next PyTorch-specific concern is ensuring that the model and its tensors perform this work on the intended device.


## Device management

Every tensor and every model parameter lives on a device. PyTorch uses the CPU by default, while supported accelerators can execute many tensor operations in parallel. Accelerator speedups depend on the model, batch size, hardware, and data pipeline; small workloads may not benefit.

PyTorch does not automatically move related objects to the same device. The model parameters, inputs, targets, and any tensors used in the same operation must agree. Otherwise, training fails with a device-mismatch error.

A common CUDA-aware setup is:

1. Choose CUDA when an NVIDIA GPU is available, otherwise choose CPU.
2. Move the model once before training.
3. Move every input and target batch inside the loop.
4. Let the model produce outputs on the same device as its parameters.

Apple systems may also support the `mps` device, but CUDA and CPU are the environments used here.


In [9]:
device: torch.device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
device_model: DigitClassifier = DigitClassifier().to(device)
device_images: torch.Tensor = images.to(device)
device_labels: torch.Tensor = labels.to(device)

# A model's parameters carry the device; the module has no .device attribute.
model_device: torch.device = next(device_model.parameters()).device
device_logits: torch.Tensor = device_model(device_images)

print(f"selected device: {device}")
print(f"model parameters: {model_device}")
print(f"inputs: {device_images.device}")
print(f"targets: {device_labels.device}")
print(f"outputs: {device_logits.device}")

selected device: cuda
model parameters: cuda:0
inputs: cuda:0
targets: cuda:0
outputs: cuda:0


### `.to()` assignment and the training pattern

For tensors, `.to(device)` returns a tensor on the requested device; it does not modify the original tensor in place. Reassign the result:

```python
inputs = inputs.to(device)
targets = targets.to(device)
```

By contrast, `nn.Module.to(device)` moves the module's registered parameters and buffers and returns the module itself. Assigning the result remains a clear and consistent style.

A device-aware training loop keeps the familiar order while adding the two batch transfers before the forward pass.


In [10]:
device_optimizer: optim.SGD = optim.SGD(device_model.parameters(), lr=0.01)
device_loss_function: nn.CrossEntropyLoss = nn.CrossEntropyLoss()
device_model.train()

for batch_images, batch_labels in train_loader:
    # Move each new batch to the model's device.
    batch_images = batch_images.to(device)
    batch_labels = batch_labels.to(device)

    device_optimizer.zero_grad()
    batch_logits: torch.Tensor = device_model(batch_images)
    batch_loss: torch.Tensor = device_loss_function(batch_logits, batch_labels)
    batch_loss.backward()
    device_optimizer.step()

    # One batch is enough to demonstrate the device-aware pattern.
    break

print(f"device-aware training loss: {batch_loss.item():.4f}")

device-aware training loss: 2.3047


### Accelerator memory and batch size

Accelerator memory is limited, and a batch must fit alongside the model parameters, gradients, activations, and optimizer state. Batch size therefore creates a practical trade-off:

- Smaller batches use less memory but may process the dataset less efficiently.
- Larger batches can improve throughput until they exceed available memory.
- A batch size around 32 or 64 is a common experiment to start with, not a universal optimum.

If CUDA reports an out-of-memory error, lowering the batch size is usually the first fix. When debugging any device problem, print the devices of the model parameters, inputs, targets, and outputs before changing anything else.
